In [0]:
# Read the bronze table
df = spark.table("workspace.default.diabetic_data")

print("Shape:", df.count(), "rows", len(df.columns), "columns")
df.show(5)

In [0]:
from pyspark.sql.functions import col, when

# Drop columns with too many missing values
df = df.drop("weight", "payer_code", "medical_specialty")

# Remove invalid gender
df = df.filter(df.gender != "Unknown/Invalid")

# Replace ? with Unknown
for col_name in ["race", "diag_1", "diag_2", "diag_3"]:
    df = df.withColumn(col_name, 
         when(col(col_name) == "?", "Unknown").otherwise(col(col_name)))

# Keep only first encounter per patient
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("patient_nbr").orderBy("encounter_id")
df = df.withColumn("row_num", row_number().over(window))
df = df.filter(df.row_num == 1).drop("row_num")

print("Silver layer shape:", df.count(), "rows", len(df.columns), "columns")
df.show(5)

In [0]:
# Save as Delta table - this is the Silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_diabetic")

print("✅ Silver Delta table saved: workspace.default.silver_diabetic")

# Verify
silver = spark.table("workspace.default.silver_diabetic")
print("Rows:", silver.count())
print("Columns:", len(silver.columns))